<a href="https://colab.research.google.com/github/niola33/OutamationWork-/blob/main/Chatbot_with_gradio_fully.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio pymupdf pytesseract pillow sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.2 MB/s eta 0:00:00


In [2]:
import fitz
import gradio as gr
import numpy as np
import faiss
import pytesseract
import torch

from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

if torch.cuda.is_available():
    model = model.to("cuda")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [16]:
chunks = []
metadata = []
index = None
LAST_ANSWER= ""
LAST_SOURCES = []
FEEDBACK_LOG= []


In [5]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        # OCR fallback
        if not text.strip():
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img)

        pages.append((page_num + 1, text))

    return pages

In [6]:
def chunk_text(text, size=500, overlap=100):
    parts = []
    start = 0

    while start < len(text):
        end = start + size
        parts.append(text[start:end])
        start += size - overlap

    return parts

In [12]:
def process_pdf(file):
    global chunks, metadata, index

    chunks = []
    metadata = []

    pages = extract_text(file.name)

    for page_num, text in pages:
        page_chunks = chunk_text(text)

        for c in page_chunks:
            chunks.append(c)
            metadata.append({
                "page": page_num,
                "source": file.name
            })

    embeddings = embed_model.encode(chunks)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))

    return f"Processed {len(pages)} pages and created {len(chunks)} chunks."

In [13]:
def retrieve(query, k=4):
    query_vec = embed_model.encode([query])
    distances, ids = index.search(np.array(query_vec), k)

    results = []

    for i in ids[0]:
        results.append({
            "text": chunks[i],
            "meta": metadata[i]
        })

    return results

In [14]:
def build_prompt(query, retrieved):
    context = ""
    for r in retrieved:
        context += f"[Source: page {r['meta']['page']}]\n{r['text']}\n\n"

    prompt = f"""
You are a mortgage document question-answering assistant.

Answer the user's question using ONLY the context below.
If the answer is not clearly supported by the context, say:
"I could not find enough information in the uploaded documents."

Rules:
- Do not use outside knowledge.
- Be concise and factual.
- Cite the relevant page numbers at the end.
- If multiple chunks disagree, mention the ambiguity.

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [8]:
def generate_answer(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [17]:
def answer_query(query):
    global LAST_ANSWER, LAST_SOURCES

    retrieved = retrieve(query)

    if not retrieved:
        LAST_ANSWER = "I couldn't find relevant information in the uploaded documents."
        LAST_SOURCES = []
        return {
            "answer": LAST_ANSWER,
            "sources": [],
            "confidence": 0.0,
            "chunks_used": 0
        }

    prompt = build_prompt(query, retrieved)
    result = generate_answer(prompt)

    confidence = 1 / (1 + len(retrieved))
    sources = [f"Page {r['meta']['page']}" for r in retrieved]

    LAST_ANSWER = result
    LAST_SOURCES = sources

    return {
        "answer": result,
        "sources": sources,
        "confidence": round(confidence, 2),
        "chunks_used": len(retrieved)
    }

In [18]:
def chat_fn(message, history):
    result = answer_query(message)

    formatted_answer = f"""
{result['answer']}

**Sources:** {", ".join(result['sources']) if result['sources'] else "None"}
**Confidence:** {result['confidence']}
**Chunks Used:** {result['chunks_used']}
"""

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": formatted_answer}
    ]

    return history, ""

In [19]:
def set_thinking():
    return "Thinking..."

def clear_thinking():
    return ""

def handle_feedback(label):
    global FEEDBACK_LOG, LAST_ANSWER, LAST_SOURCES

    if not LAST_ANSWER:
        return "No answer available yet to rate."

    FEEDBACK_LOG.append({
        "feedback": label,
        "answer": LAST_ANSWER,
        "sources": LAST_SOURCES
    })

    return f"Feedback recorded: {label}"

In [20]:
import json

def save_feedback_log():
    global FEEDBACK_LOG

    if not FEEDBACK_LOG:
        return "No feedback to save."

    with open("feedback_log.json", "w") as f:
        json.dump(FEEDBACK_LOG, f, indent=2)

    return "Feedback saved to feedback_log.json"

In [21]:
with gr.Blocks(
    title="Mortgage Document Q&A Assistant",
    theme=gr.themes.Soft(
        primary_hue="green",
        secondary_hue="emerald",
        neutral_hue="stone"
    ),
    css="""
    body {
        background: linear-gradient(135deg, #0b1f14, #123524, #1b4332);
    }

    .gradio-container {
        background: transparent !important;
    }

    .panel-card {
        background: rgba(255,255,255,0.06) !important;
        border: 1px solid rgba(255,255,255,0.10) !important;
        border-radius: 20px !important;
        box-shadow:
            0 10px 25px rgba(0,0,0,0.35),
            inset 0 1px 0 rgba(255,255,255,0.08) !important;
        backdrop-filter: blur(10px);
        padding: 18px !important;
    }

    .green-title {
        text-align: center;
        color: #d8f3dc;
        font-size: 2.1rem;
        font-weight: 800;
        text-shadow: 0 2px 12px rgba(0,0,0,0.45);
        margin-bottom: 0.2rem;
    }

    .green-subtitle {
        text-align: center;
        color: #b7e4c7;
        font-size: 1rem;
        margin-bottom: 1rem;
    }

    button {
        border-radius: 14px !important;
        box-shadow: 0 6px 18px rgba(0,0,0,0.28) !important;
        transition: transform 0.15s ease, box-shadow 0.15s ease !important;
    }

    button:hover {
        transform: translateY(-2px);
        box-shadow: 0 10px 24px rgba(0,0,0,0.35) !important;
    }

    textarea, input {
        border-radius: 14px !important;
    }

    .thinking-box textarea {
        color: #d8f3dc !important;
        font-weight: 700 !important;
    }
    """
) as demo:

    gr.Markdown(
        """
        <div class="green-title">🌿 Mortgage Document Q&A Assistant</div>
        <div class="green-subtitle">
        Upload mortgage PDFs, process them, and chat with grounded answers from your documents.
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1, elem_classes="panel-card"):
            pdf = gr.File(
                label="📄 Upload Mortgage PDF",
                file_types=[".pdf"]
            )

            process_btn = gr.Button("⚙️ Process Document", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)

            thinking_status = gr.Textbox(
                label="System Status",
                value="Ready",
                interactive=False,
                elem_classes="thinking-box"
            )

            save_feedback_btn = gr.Button("💾 Save Feedback Log")
            feedback_save_status = gr.Textbox(label="Feedback Save Status", interactive=False)

        with gr.Column(scale=2, elem_classes="panel-card"):
            chatbot = gr.Chatbot(label="Chat History", height=520, type="messages")

            msg = gr.Textbox(
                placeholder="Ask a question about your mortgage documents...",
                label="Your Question"
            )

            with gr.Row():
                send = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")

            gr.Markdown("### Was this answer good?")
            with gr.Row():
                thumbs_up = gr.Button("👍")
                thumbs_down = gr.Button("👎")

            feedback_status = gr.Textbox(label="Feedback Status", interactive=False)

    process_btn.click(
        process_pdf,
        inputs=pdf,
        outputs=status
    )

    send.click(
        fn=set_thinking,
        inputs=None,
        outputs=thinking_status
    ).then(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    ).then(
        fn=lambda: "Ready",
        inputs=None,
        outputs=thinking_status
    )

    msg.submit(
        fn=set_thinking,
        inputs=None,
        outputs=thinking_status
    ).then(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    ).then(
        fn=lambda: "Ready",
        inputs=None,
        outputs=thinking_status
    )

    clear_btn.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot
    )

    thumbs_up.click(
        fn=lambda: handle_feedback("Thumbs Up"),
        inputs=None,
        outputs=feedback_status
    )

    thumbs_down.click(
        fn=lambda: handle_feedback("Thumbs Down"),
        inputs=None,
        outputs=feedback_status
    )

    save_feedback_btn.click(
        fn=save_feedback_log,
        inputs=None,
        outputs=feedback_save_status
    )

demo.launch(share=True)

/tmp/ipykernel_446/2653296670.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_446/2653296670.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_446/2653296670.py:96: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=520, type="messages")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0939574637fc07a7f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
